In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.vectorstores import FAISS     
from langchain import hub 
import os 
from dotenv import load_dotenv

load_dotenv()
api_key=os.getenv("bOTAPIKEY")

llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash", api_key=api_key)

loader= WebBaseLoader("https://python.langchain.com/docs/concepts/")
docs=loader.load()


splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunked_document=splitter.split_documents(docs)

embeddings=GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=api_key)

vectordb=FAISS.from_documents(chunked_document, embeddings)

retriever=vectordb.as_retriever()

retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C0A439FE20>, search_kwargs={})

In [13]:
from langchain.tools import WikipediaQueryRun
from langchain.utilities import WikipediaAPIWrapper

In [14]:
api_wrapper= WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
tool=WikipediaQueryRun(api_wrapper=api_wrapper)

In [15]:
tool

WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\langchain\\Langchain\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200))

In [ ]:
from langchain.tools.retriever import create_retriever_tool
retiriever_tool=create_retriever_tool(retriever, "langchain_search", 
                      "search for information about langchain. For any question about langcahin use this tool")

retiriever_tool.name

'langchain_search'

In [26]:
from langchain.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=100)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [27]:
tools=[tool, arxiv, retiriever_tool]

In [28]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\langchain\\Langchain\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=100)),
 Tool(name='langchain_search', description='search for information about langchai. For any question about langcahin use this tool', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x000001C080C54EE0>, retriever=VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vect

In [29]:
prompt= hub.pull("hwchase17/openai-functions-agent")
prompt.messages

d:\langchain\Langchain\venv\lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [30]:
#Agents
from langchain.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm, tools, prompt)



In [ ]:
#Agent excuter 
from langchain.agents import AgentExecutor
agent_executor=AgentExecutor(agent= agent ,tools = tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [35]:
agent_executor.invoke({"input":"Teel me about langchain"})



> Entering new AgentExecutor chain...

Invoking: `langchain_search` with `{'query': 'tell me about langchain'}`


callingToolsTracingVector storesWhy LangChain?Ecosystem🦜🛠️ LangSmith🦜🕸️ LangGraphVersionsv0.3v0.2Pydantic compatibilityMigrating from v0.0 chainsHow to migrate from v0.0 chainsMigrating from ConstitutionalChainMigrating from ConversationalChainMigrating from ConversationalRetrievalChainMigrating from LLMChainMigrating from LLMMathChainMigrating from LLMRouterChainMigrating from MapReduceDocumentsChainMigrating from MapRerankDocumentsChainMigrating from MultiPromptChainMigrating from RefineDocumentsChainMigrating from RetrievalQAMigrating from StuffDocumentsChainUpgrading to LangGraph memoryHow to migrate to LangGraph memoryHow to use BaseChatMessageHistory with LangGraphMigrating off ConversationBufferMemory or ConversationStringBufferMemoryMigrating off ConversationBufferWindowMemory or ConversationTokenBufferMemoryMigrating off ConversationSummaryMemory or ConversationS

{'input': 'Teel me about langchain',
 'output': 'LangChain is a framework designed to help build applications using large language models. It provides tools and components for various tasks, including:\n\n* **Ecosystem:** Offers integrations and tools like LangSmith and LangGraph.\n* **Versions:** Mentions different versions and compatibility.\n* **Migration Guides:** Provides guides for migrating between different versions and components.\n* **Conceptual Guide:** Explains key concepts behind the LangChain framework and AI applications.\n* **Tutorials:** Offers tutorials for building various applications like question answering systems, chatbots, and more.\n* **How-to Guides:** Provides guides on specific tasks like using tools in a chain, adding memory to chatbots, and streaming chat model responses.'}

In [36]:
agent_executor.invoke({"input":"Teel me about Mchine leraning"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Machine learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and genMachine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data.

> Finished chain.


{'input': 'Teel me about Mchine leraning',
 'output': 'Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data.'}

In [37]:
agent_executor.invoke({"input":"whats the paper 1605.08386 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1605.08386'}`


Published: 2016-05-26
Title: Heat-bath random walks with Markov bases
Authors: Caprice Stanley, TobiThe paper 1605.08386, titled "Heat-bath random walks with Markov bases", was published on 2016-05-26 and is authored by Caprice Stanley and Tobi.

> Finished chain.


{'input': 'whats the paper 1605.08386 about?',
 'output': 'The paper 1605.08386, titled "Heat-bath random walks with Markov bases", was published on 2016-05-26 and is authored by Caprice Stanley and Tobi.'}